# Logistic Regression and Regularization


## Introduction

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/11-Regression-II-Logistic-Regularization.ipynb)

In [ ]:
#| echo: false
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import matplotlib as mp
import sklearn
from IPython.display import Image, HTML
import statsmodels.api as sm
from sklearn import model_selection
from sklearn import metrics

import laUtilities as ut

%matplotlib inline

So far we have seen linear regression: a continuous valued observation is
estimated as a linear (or affine) function of the independent variables.

This lecture has two parts:

* **Part 1 -- Logistic regression.** The observation is a 0/1 variable and we
  want a *probability*. We fit a linear model to the log-odds, invert it through
  the sigmoid, fit by maximum likelihood, and then evaluate the result as a
  classifier.
* **Part 2 -- Regularization.** When the columns of $X$ are nearly dependent the
  least-squares coefficients become large and unstable. A penalty on the size of
  $\boldsymbol{\beta}$ -- ridge or LASSO -- fixes that, for linear and logistic
  regression alike.


# Part 1: Logistic Regression

## Estimating a Probability


* Imagine that you are observing a binary variable -- value 0 or 1.

* That is, these could be pass/fail, admit/reject, Democrat/Republican, etc.

* Assume there is some __probability__ of observing a 1, and that probability is a
function of certain independent variables.

* So the key properties of a problem that make it appropriate for logistic
regression are:

    * You are trying to predict a __categorical__ variable
    * You want to estimate a __probability__ of seeing a particular value of the
      categorical variable.


## Example: Grad School Admission


**Note**

The following example was adapted from this
[URL](http://www.ats.ucla.edu/stat/r/dae/logit.htm) which seems to be no longer
available. There is an
[archive of the page](https://web.archive.org/web/20161118221128/http://www.ats.ucla.edu/stat/r/dae/logit.htm)
and an [archive of the dataset](https://web.archive.org/web/20161022051727/http://www.ats.ucla.edu/stat/data/binary.csv).


Let's consider this question:

> What is the probability I will be admitted to Grad School?

Let's see how variables, such as,

* _GRE_ (Graduate Record Exam scores),
* _GPA_ (grade point average), and
* prestige of the undergraduate institution

affect admission into graduate school.


The response variable, admit/don't admit, is a binary variable.

So there are three predictor variables: __gre,__ __gpa__ and __rank.__

* We will treat the variables _gre_ and _gpa_ as continuous.
* The variable _rank_ takes on the values 1 through 4 with 1 being the highest prestige.

**Rank is ordinal, and we feed it in as a number**

Treating _rank_ as a number assumes that each step down in prestige has the
*same* effect on the log-odds. The alternative is one indicator column per rank
(`C(rank)` in formula syntax), which costs two more coefficients. We keep the
numeric version here; the categorical version is shown in
[Appendix: Regression Formulas with statsmodels](A8-Statsmodels-Formulas.qmd).



Let's look at 10 lines of the data:

In [ ]:
#| echo: false
# original data source: http://www.ats.ucla.edu/stat/data/binary.csv
df = pd.read_csv('data/ats-admissions.csv')
df.head(10)

In [ ]:
df.shape

and some summary statistics:

In [ ]:
#| code-fold: true
df.describe()

We can also plot histograms of the variables:

In [ ]:
#| code-fold: true
df.hist(figsize = (10, 4));

> Note how `df.hist()` automatically plots a histogram for each column in the
> dataframe as a subplot.


Let's look at how each independent variable affects admission probability by
plotting the mean admission probability as a function of the independent variable.

> Note that there's a greatly expanded [`groupby`](A3-Pandas.qmd#understanding-pandas-dataframe.groupby) section in the Pandas refresher.

We add error bars to the means to indicate the standard error of the mean.


First, __rank__:

In [ ]:
#| code-fold: true
# Calculate mean and standard error
grouped = df.groupby('rank')['admit']
means = grouped.mean()

# Compute 'standard error of the mean'
errors = grouped.std() / np.sqrt(grouped.count())

# Plot with error bars
ax = means.plot(marker='o', yerr=errors, fontsize=12, capsize=5)
ax.set_ylabel('P[admit]', fontsize=16)
ax.set_xlabel('Rank', fontsize=16);

Next, __GRE__:

In [ ]:
#| code-fold: true
grouped_gre = df.groupby('gre')['admit']
means_gre = grouped_gre.mean()
errors_gre = grouped_gre.std() / np.sqrt(grouped_gre.count())

ax = means_gre.plot(marker='o', yerr=errors_gre, fontsize=12, capsize=5)
ax.set_ylabel('P[admit]', fontsize=16)
ax.set_xlabel('GRE', fontsize=16);

Finally, __GPA__ (for this visualization, we aggregate GPA into 7 equal-width bins
and plot each bin's mean at the bin centre):

In [ ]:
#| code-fold: true
bins = np.linspace(df.gpa.min(), df.gpa.max(), 8)
gpa_bin = pd.cut(df.gpa, bins, include_lowest=True)
grouped_gpa = df.groupby(gpa_bin, observed=True)['admit']
means_gpa = grouped_gpa.mean()
errors_gpa = grouped_gpa.std() / np.sqrt(grouped_gpa.count())
bin_centers = [interval.mid for interval in means_gpa.index]

ax = plt.figure(figsize=(6, 4)).add_subplot()
ax.errorbar(bin_centers, means_gpa, yerr=errors_gpa, marker='o', capsize=5)
ax.tick_params(labelsize=12)
ax.set_ylabel('P[admit]', fontsize=16)
ax.set_xlabel('GPA (bin centre)', fontsize=16);

Finally, we plot admission status versus GRE score for each data point for each of the four ranks:

In [ ]:
#| code-fold: true
df1 = df[df['rank']==1]
df2 = df[df['rank']==2]
df3 = df[df['rank']==3]
df4 = df[df['rank']==4]

fig = plt.figure(figsize = (10, 5))

ax1 = fig.add_subplot(221)
df1.plot.scatter('gre','admit', ax = ax1)
plt.title('Rank 1 Institutions')

ax2 = fig.add_subplot(222)
df2.plot.scatter('gre','admit', ax = ax2)
plt.title('Rank 2 Institutions')

ax3 = fig.add_subplot(223, sharex = ax1)
df3.plot.scatter('gre','admit', ax = ax3)
plt.title('Rank 3 Institutions')

ax4 = fig.add_subplot(224, sharex = ax2)
plt.title('Rank 4 Institutions')
df4.plot.scatter('gre','admit', ax = ax4);

What we want to do is to fit a model that predicts the probability of admission
as a function of these independent variables.

## Logistic Regression

* Logistic regression is concerned with estimating a __probability.__

* However, all that is available are categorical observations, which we will code as 0/1.

* That is, these could be pass/fail, admit/reject, Democrat/Republican, etc.

* Now, a linear function like $\beta_0 + \beta_1 x$ cannot be used to predict
probability directly, because
    * the linear function takes on all values (from -$\infty$ to +$\infty$),
    * and probability only ranges over $[0, 1]$.


## Odds and Log-Odds

However, there is a transformation of probability that works: it is called
__log-odds__.

For any probability $p$, the __odds__ is defined as $p/(1-p)$, which is the ratio
of the probability of an event to the probability of the non-event.

Notice that odds vary from 0 to $\infty$, and odds < 1 indicates that $p < 1/2$.

Odds are still not a good target for a *linear* function, for two reasons.

* **Range.** A line takes negative values; odds cannot. Taking the log maps
  $(0, \infty)$ onto the whole real line, so nothing the line predicts is impossible.
* **Symmetry.** Odds treat the two outcomes asymmetrically: $p = 0.9$ gives odds
  $9$ while $p = 0.1$ gives odds $1/9$. Log-odds are symmetric: $+2.2$ and $-2.2$.
  Swapping which outcome we call "1" just flips the sign of every coefficient.



There is a third reason, about what a linear model *means*: if the log-odds are
linear in $x$, then each unit increase in $x$ **multiplies the odds by the same
factor** $e^{\beta_1}$, whatever the starting point. That is how we usually think
of a risk factor acting -- "doubles your odds" -- and it is the reading of the
coefficients we will use once we have fit a model.

The log-odds, $\log p/(1-p)$, is also called the __logit__ function, an
abbreviation for **log**istic un**it**.

In [ ]:
#| code-fold: true
pvec = np.linspace(0.01, 0.99, 100)
ax = plt.figure(figsize = (6, 4)).add_subplot()
ax.plot(pvec, np.log(pvec / (1-pvec)))
ax.tick_params(labelsize=12)
ax.set_xlabel('Probability', fontsize = 14)
ax.set_ylabel('Log-Odds', fontsize = 14)
ax.set_title(r'Logit Function: $\log (p/(1-p))$', fontsize = 16);

So, logistic regression does the following: it does a linear regression of
$\beta_0 + \beta_1 x$ against $\log p/(1-p)$.

That is, it fits:

$$
\begin{aligned}
\beta_0 + \beta_1 x &= \log \frac{p(x)}{1-p(x)} \\
e^{\beta_0 + \beta_1 x} &= \frac{p(x)}{1-p(x)} \quad \text{(exponentiate both sides)} \\
e^{\beta_0 + \beta_1 x} (1-p(x)) &= p(x) \quad \text{(multiply both sides by $1-p(x)$)} \\
e^{\beta_0 + \beta_1 x}  &= p(x) + p(x)e^{\beta_0 + \beta_1 x} \quad \text{(distribute $p(x)$)} \\
\frac{e^{\beta_0 + \beta_1 x}}{1 +e^{\beta_0 + \beta_1 x}} &= p(x)
\end{aligned}
$$


So, logistic regression fits a probability of the following form:

$$
p(x) = P(y=1\mid x) = \frac{e^{\beta_0+\beta_1 x}}{1+e^{\beta_0+\beta_1 x}}.
$$

This is a **sigmoid function**; when $\beta_1 > 0$,

* as $x\rightarrow \infty$, then $p(x)\rightarrow 1$ and
* as $x\rightarrow -\infty$, then $p(x)\rightarrow 0$.


The sigmoid comes up so often that it gets its own symbol:

$$
\sigma(z) = \frac{e^{z}}{1+e^{z}} = \frac{1}{1+e^{-z}},
\qquad\text{so}\qquad
p(x) = \sigma(\beta_0 + \beta_1 x).
$$

With $p$ features we collect them, together with a leading $1$ for the intercept,
into $\mathbf{x} = (1, x_1, \ldots, x_p)^T$ and the coefficients into
$\boldsymbol{\beta} = (\beta_0, \beta_1, \ldots, \beta_p)^T$. The model is then

$$
p(\mathbf{x}) = \sigma(\boldsymbol{\beta}^T\mathbf{x}),
\qquad
\text{logit}\, p(\mathbf{x}) = \boldsymbol{\beta}^T\mathbf{x}.
$$

Everything below is written for one feature; replace $\beta_0 + \beta_1 x$ by
$\boldsymbol{\beta}^T\mathbf{x}$ and it holds for any number of features.


In [ ]:
#| code-fold: true
#| fig-align: center
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.linspace(0, 50, 200)
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

# left: beta_0 fixed, beta_1 varies
for b1 in [0.2, 0.4, 0.6, 1.0]:
    axs[0].plot(x, sigmoid(-8 + b1 * x), label=rf'$\beta_1 = {b1}$')
axs[0].set_title(r'$\beta_0 = -8$ fixed: $\beta_1$ sets the steepness', fontsize=13)

# right: beta_1 fixed, beta_0 varies; mark where p = 1/2
for b0 in [-4, -8, -12, -16]:
    line, = axs[1].plot(x, sigmoid(b0 + 0.4 * x), label=rf'$\beta_0 = {b0}$')
    axs[1].plot(-b0 / 0.4, 0.5, 'o', color=line.get_color())
axs[1].axhline(0.5, color='gray', linestyle=':', linewidth=1)
axs[1].set_title(r'$\beta_1 = 0.4$ fixed: $\beta_0$ shifts the curve', fontsize=13)

for ax in axs:
    ax.set_xlabel('$x$', fontsize=13)
    ax.tick_params(labelsize=11)
    ax.legend(fontsize=11)
axs[0].set_ylabel('$p(x)$', fontsize=13)
plt.tight_layout()

* $\beta_1$ controls how fast $p(x)$ rises from $0$ to $1$ (and its sign sets the direction).
* The curve crosses $p = 1/2$ where the log-odds are zero, at $x = -\beta_0/\beta_1$;
  increasing $\beta_0$ shifts the curve to the left (the dots on the right panel).



$\beta_0$ is the **log-odds at $x = 0$**: $p(0) = \sigma(\beta_0)$.

It is tempting to call this the *base rate* -- the overall fraction of 1s in the
data -- but that is only true if the features have been **centered**, so that
$x = 0$ is the average data point. In the admissions data $x = 0$ means GRE 0,
GPA 0.0, rank 0 -- a student who does not exist -- and we will see that
$\sigma(\hat\beta_0)$ is nowhere near the admit rate.


The function $f(x) = \log (x/(1-x))$ is called the __logit__ function.

So a compact way to describe logistic regression is that it finds regression
coefficients $\beta_0, \beta_1$ to fit:

$$
\text{logit}\left(p(x)\right)=\log\left(\frac{p(x)}{1-p(x)} \right) = \beta_0 + \beta_1 x.
$$

Note also that the __inverse__ logit function is:

$$
\text{logit}^{-1}(z) = \frac{e^z}{1 + e^z} = \sigma(z).
$$

Somewhat confusingly, this is called the __logistic__ function.


So, the best way to think of logistic regression is that we compute a linear function:

$$
\beta_0 + \beta_1 x,
$$

and then *map* that to a probability using the inverse $\text{logit}$ function:

$$
\sigma(\beta_0 + \beta_1 x) = \frac{e^{\beta_0+\beta_1 x}}{1+e^{\beta_0+\beta_1 x}}.
$$

## Logistic vs Linear Regression

Let's take a moment to compare linear and logistic regression.

In __Linear regression__ we fit

$$
y_i = \beta_0 +\beta_1 x_i + \epsilon_i.
$$

We do the fitting by minimizing the sum of squared errors $\Vert\epsilon\Vert^2$.
This can be done in closed form using either geometric arguments or by calculus.

Now, if $\epsilon_i$ comes from a normal distribution with mean zero and some
fixed variance, then minimizing the sum of squared errors is exactly the same as finding the
maximum likelihood of the data with respect to the probability of the errors.

So, in the case of linear regression, it is a lucky fact that the __MLE__ of
$\beta_0$ and $\beta_1$ can be found by a __closed-form__ calculation.


In __Logistic regression__ we fit

$$
\text{logit}(p(x_i)) = \beta_0 + \beta_1 x_i.
$$


with $\text{P}(y_i=1\mid x_i)=p(x_i).$

How should we choose parameters?

Here too, we use Maximum Likelihood Estimation of the parameters.

That is, we choose the parameter values that maximize the likelihood of the data given the model.

$$
\text{P}(y_i \mid x_i) =
\left\{\begin{array}{lr}\sigma(\beta_0 + \beta_1 x_i)& \text{if } y_i = 1\\
1 - \sigma(\beta_0 + \beta_1 x_i)& \text{if } y_i = 0\end{array}\right.
$$


We can write this as a single expression. Writing $p_i = \sigma(\beta_0 + \beta_1 x_i)$,

$$
\text{P}(y_i \mid x_i) = p_i^{\,y_i} (1-p_i)^{1-y_i},
$$

where we assume the parameters are fixed.

We can reinterpret this to express the __likelihood__ of parameters $\beta_0$, $\beta_1$
given that we have observed the data $(x_i, y_i)$:

$$
L(\beta_0, \beta_1 \mid x_i, y_i) = p_i^{\,y_i} (1-p_i)^{1-y_i}.
$$

That is one observation. We have $n$ of them.

## From one observation to the whole data set

The observations are independent, so the likelihood of the data set is the **product**
and its logarithm -- the **log-likelihood** -- is the sum:

$$
L(\boldsymbol{\beta}) = \prod_{i=1}^n p_i^{\,y_i}(1-p_i)^{1-y_i}
\qquad\Longrightarrow\qquad
\ell(\boldsymbol{\beta}) = \sum_{i=1}^n \big[\, y_i \log p_i + (1-y_i)\log(1-p_i) \,\big].
$$

* $-\ell(\boldsymbol{\beta})/n$ is the **log-loss** (also called *cross-entropy*):
  the quantity `sklearn` and every neural-network library report. Maximizing
  $\ell$ and minimizing log-loss are the same thing.
* **This is our objective function to maximize.** Unlike least squares it is not
  quadratic in $\boldsymbol{\beta}$, so setting the gradient to zero gives no
  closed form -- we climb numerically.



The sigmoid has a derivative that is easy to remember:

$$
\sigma'(z) = \sigma(z)\,\big(1 - \sigma(z)\big).
$$

Using it, with $p_i = \sigma(\boldsymbol{\beta}^T\mathbf{x}_i)$, the chain rule
gives $\partial p_i/\partial\boldsymbol{\beta} = p_i(1-p_i)\,\mathbf{x}_i$, and the
$p_i(1-p_i)$ cancels against the denominators of $\log p_i$ and $\log(1-p_i)$:

$$
\nabla \ell(\boldsymbol{\beta})
= \sum_{i=1}^n (y_i - p_i)\,\mathbf{x}_i
= X^T(\mathbf{y} - \mathbf{p}),
$$

where $X$ is the $n \times (p+1)$ matrix with rows $\mathbf{x}_i^T$ and
$\mathbf{p} = \sigma(X\boldsymbol{\beta})$ applied entrywise.

**Residual times feature** -- the same shape as the normal equations
$X^T(\mathbf{y} - X\boldsymbol{\beta}) = 0$ from linear regression, with a sigmoid
inside. The difference is that $\mathbf{p}$ depends on $\boldsymbol{\beta}$
nonlinearly, so we cannot solve $\nabla\ell = 0$ directly.

## Gradient ascent on the log-likelihood

**Algorithm:**

1. **Initialize** parameters: Start with random values $\boldsymbol{\beta}^{(0)}$

2. **Compute gradient**: Calculate $\nabla \ell(\boldsymbol{\beta}^{(t)}) = X^T(\mathbf{y} - \mathbf{p}^{(t)})$ -- the direction of steepest increase

3. **Update parameters**: Take a step in that direction:
   $$\boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} + \eta \nabla \ell(\boldsymbol{\beta}^{(t)})$$
   where $\eta$ is the **learning rate** (step size; the same $\eta$ as in Lecture 16)

4. **Repeat** steps 2-3 until convergence (gradient $\approx 0$ or max iterations reached)

**Result:** Parameters that maximize the likelihood of the observed data. The log-likelihood of logistic regression is concave, so this maximum is global.

We are climbing $\ell$, so this is gradient *ascent*; the libraries minimize the
log-loss instead, which is gradient *descent* on $-\ell$ -- the same steps.
Gradient descent gets a full treatment in
[Neural Networks I](16-Neural-Networks-I.qmd#gradient-descent).


Three observations, one feature: $x = (-1, 0, 2)$, $y = (0, 0, 1)$, so
$\mathbf{x}_i = (1, x_i)^T$. Start at $\boldsymbol{\beta}^{(0)} = (0, 0)^T$ with $\alpha = 0.1$.

* $\boldsymbol{\beta}^T\mathbf{x}_i = 0$ for every $i$, so $\mathbf{p} = (0.5, 0.5, 0.5)^T$
  and $\mathbf{y} - \mathbf{p} = (-0.5, -0.5, 0.5)^T$.
* $\partial\ell/\partial\beta_0 = \sum_i (y_i - p_i) = -0.5$;
  $\;\partial\ell/\partial\beta_1 = \sum_i (y_i - p_i)\,x_i = 0.5 + 0 + 1.0 = 1.5$.
* $\boldsymbol{\beta}^{(1)} = (0, 0) + 0.1\,(-0.5, 1.5) = (-0.05, 0.15)$.


The intercept moves down (two of three labels are 0) and the slope moves up (the
1 sits at the largest $x$). In today's activity you will code exactly this loop
and check that it converges to what `sklearn` reports.


## Logistic Regression In Practice

So, in summary, we have:

**Input** pairs $(\mathbf{x}_i,y_i)$

**Output** parameters $\widehat{\boldsymbol{\beta}}$ that maximize the
likelihood of the data given these parameters for the logistic regression model.

**Method** Maximum likelihood estimation, obtained by gradient ascent on $\ell$
(the packages use a faster second-order variant of the same idea).

The standard package will give us a coefficient $\beta_j$ for each
independent variable (feature).


If we want to include a constant (i.e., $\beta_0$) we need to add a column of 1s (just
like in linear regression).

In [ ]:
#| code-fold: true
df['intercept'] = 1.0
train_cols = df.columns[1:]
train_cols

In [ ]:
#| code-fold: true
logit = sm.Logit(df['admit'], df[train_cols])

# fit the model
result = logit.fit()

Statsmodels gives us a summary of the model fit.

In [ ]:
#| code-fold: true
result.summary()

All three predictors are significant at the 5% level: no 95% confidence interval
contains zero. The GRE interval is displayed as `[0.000, 0.004]` only because of
rounding; unrounded it is

In [ ]:
#| code-fold: true
result.conf_int().loc['gre']

## Reading the coefficients

The coefficients live on the **log-odds** scale, and $e^{\beta_j}$ is the factor
by which the odds are multiplied when $x_j$ increases by one unit, holding the
other features fixed.

In [ ]:
#| code-fold: true
coef_table = pd.DataFrame({
    'coefficient': result.params,
    'odds ratio e^beta': np.exp(result.params),
})
coef_table.loc['gre (per 100 points)'] = [100 * result.params['gre'],
                                          np.exp(100 * result.params['gre'])]
coef_table.round(3)

* **GPA:** $\hat\beta = 0.777$, so one more GPA point multiplies the odds of admission
  by $e^{0.777} \approx 2.17$ -- it more than doubles them.
* **GRE:** $\hat\beta = 0.0023$ per point, which is $0.23$ per 100 points, so 100 more
  GRE points multiply the odds by $e^{0.23} \approx 1.26$.
* **Rank:** $\hat\beta = -0.56$, so each step *down* in prestige (rank 1 to 2, 2 to 3,
  ...) multiplies the odds by $e^{-0.56} \approx 0.57$ -- nearly halves them.


In [ ]:
#| code-fold: true
b0 = result.params['intercept']
mean_applicant = [df.gre.mean(), df.gpa.mean(), df['rank'].mean(), 1.0]
print(f'intercept beta_0 = {b0:.3f};   sigma(beta_0) = {1/(1+np.exp(-b0)):.3f}')
print(f'fraction admitted in the data   = {df.admit.mean():.4f}')
print(f'p(x) at the mean GRE, GPA, rank = {result.predict([mean_applicant])[0]:.3f}')

* $\sigma(\hat\beta_0) = 3.1\%$ is the model's admission probability for a
  student with GRE 0, GPA 0.0 and rank 0. It is **not** the base rate: 31.75%
  of the applicants were admitted.
* Had we centered the features first, $x = 0$ would be the *average* applicant,
  and $\sigma(\hat\beta_0)$ would be close to the admit rate (0.30 here).


## Using the Model

Note that by fitting a model to the data, we can make predictions for inputs that
were not in the training data.

Furthermore, we can make a prediction of a probability for cases where we don't
have enough data to estimate the probability directly -- e.g., for specific
parameter values.

Let's see how well the model fits the data.


We have three independent variables, so in each case we'll use average values
for the two that we aren't evaluating.

GPA (GRE = 600, Rank = 2.5):

In [ ]:
#| code-fold: true
#| fig-align: center
gpa_grid = np.linspace(df.gpa.min(), df.gpa.max(), 50)
prob = [result.predict([600, g, 2.5, 1.0]) for g in gpa_grid]

bins = np.linspace(df.gpa.min(), df.gpa.max(), 10)
groups = df.groupby(pd.cut(df.gpa, bins, include_lowest=True), observed=True)['admit'].mean()
centers = [interval.mid for interval in groups.index]

ax = plt.figure(figsize = (7, 4)).add_subplot()
ax.plot(gpa_grid, prob)
ax.plot(centers, groups, 'o')
ax.tick_params(labelsize=12)
ax.set_xlabel('gpa', fontsize = 14)
ax.set_ylabel('P[admit]', fontsize = 14)
ax.set_title('Marginal Effect of GPA', fontsize = 16);

GRE Score (GPA = 3.4, Rank = 2.5):

In [ ]:
#| code-fold: true
#| fig-align: center
prob = [result.predict([b, 3.4, 2.5, 1.0]) for b in sorted(df.gre.unique())]
ax = plt.figure(figsize = (7, 4)).add_subplot()
ax.plot(sorted(df.gre.unique()), prob)
ax.plot(df.groupby('gre').mean()['admit'],'o')
ax.tick_params(labelsize=12)
ax.set_xlabel('gre', fontsize = 14)
ax.set_ylabel('P[admit]', fontsize = 14)
ax.set_title('Marginal Effect of GRE', fontsize = 16);

Institution Rank (GRE = 600, GPA = 3.4):

In [ ]:
#| code-fold: true
#| fig-align: center
prob = [result.predict([600, 3.4, b, 1.0]) for b in range(1,5)]
ax = plt.figure(figsize = (7, 4)).add_subplot()
ax.plot(range(1,5), prob)
ax.plot(df.groupby('rank').mean()['admit'],'o')
ax.tick_params(labelsize=12)
ax.set_xlabel('Rank', fontsize = 14)
ax.set_xlim([0.5,4.5])
ax.set_ylabel('P[admit]', fontsize = 14)
ax.set_title('Marginal Effect of Rank', fontsize = 16);

## Logistic Regression in Perspective

At the start of lecture we emphasized that logistic regression is concerned with
estimating a __probability__ model for __discrete__ (0/1) data.

However, it may well be the case that we want to do something with the
probability that amounts to __classification.__

For example, we may classify data items using a rule such as "Assign item $\mathbf{x}_i$
to Class 1 if $p(\mathbf{x}_i) > 0.5$".

For this reason, logistic regression could be considered a classification method.

## The decision boundary is still a hyperplane

The rule "$p(\mathbf{x}) > 0.5$" is the rule "$\boldsymbol{\beta}^T\mathbf{x} > 0$":
the sigmoid is monotone, so thresholding the probability is the same as
thresholding the *linear* score. The boundary $\boldsymbol{\beta}^T\mathbf{x} = 0$
is a **hyperplane** -- a line in two features -- exactly as in the linear
classifiers you have seen. The nonlinearity lives in the *confidence*, not in the
boundary.

Any other threshold $t$ slides the hyperplane parallel to itself, to
$\boldsymbol{\beta}^T\mathbf{x} = \text{logit}(t)$.


Rank-2 applicants, with the boundary $\hat{\boldsymbol{\beta}}^T\mathbf{x} = \text{logit}(t)$ drawn
for $t = 0.5$ and $t = 0.3$ (solved for GPA as a function of GRE):

In [ ]:
#| code-fold: true
#| fig-align: center
b = result.params
rank_shown = 2
gre_line = np.array([df.gre.min(), df.gre.max()])

def boundary_gpa(t):
    # beta^T x = logit(t), solved for gpa with rank fixed
    logit_t = np.log(t / (1 - t))
    return (logit_t - b['intercept'] - b['rank'] * rank_shown - b['gre'] * gre_line) / b['gpa']

shown = df[df['rank'] == rank_shown]
colors = {0: '#56B4E9', 1: '#D55E00'}   # colour-blind-safe (Okabe-Ito)
fig, ax = plt.subplots(figsize=(7, 4.5))
for label, grp in shown.groupby('admit'):
    ax.scatter(grp.gre, grp.gpa, c=colors[label], s=35, alpha=0.85,
               edgecolor='k', linewidth=0.3,
               label='admitted' if label == 1 else 'rejected')
ax.plot(gre_line, boundary_gpa(0.5), 'k-', linewidth=2, label=r'$p(\mathbf{x}) = 0.5$')
ax.plot(gre_line, boundary_gpa(0.3), 'k--', linewidth=2, label=r'$p(\mathbf{x}) = 0.3$')
ax.set_ylim(df.gpa.min() - 0.1, df.gpa.max() + 0.1)
ax.set_xlabel('GRE', fontsize=13)
ax.set_ylabel('GPA', fontsize=13)
ax.set_title('Rank-2 applicants: the boundary is a line', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.tick_params(labelsize=11)
plt.tight_layout()

Above the line the model says "admit". At $t = 0.5$ almost no rank-2 applicant
qualifies; lowering the threshold to $0.3$ slides the same line down into the
cloud. Changing the threshold never bends the boundary.


Let's use our logistic regression as a classifier.

We want to ask whether we can correctly predict whether a student gets admitted
to graduate school.

Let's separate our training and test data:

In [ ]:
#| code-fold: true
X_train, X_test, y_train, y_test = model_selection.train_test_split(
        df[train_cols], df['admit'],
        test_size=0.4, random_state=1)

Now, there are some standard metrics used when evaluating a binary classifier.

Let's say our classifier is outputting "yes" when it thinks the student will be admitted.

There are four cases:

* Classifier says "yes", and student __is__ admitted:  __True Positive.__
* Classifier says "yes", and student __is not__ admitted:  __False Positive.__
* Classifier says "no", and student __is__ admitted:  __False Negative.__
* Classifier says "no", and student __is not__ admitted:  __True Negative.__


__Accuracy__ is the fraction of all classifications that are correct:
$(\text{TP} + \text{TN}) / n$.

Accuracy has a floor set by the **prevalence** (the base rate). Only 31.75% of
these applicants were admitted, so a "classifier" that says *no* to everyone is
**68% accurate** without looking at a single feature. Under class imbalance,
accuracy mostly measures the imbalance -- which is why we need the next two
metrics.


__Precision__ is the fraction of "yes" classifications that are correct:

$$
\mbox{Precision} = \frac{\mbox{True Positives}}{\mbox{True Positives + False Positives}}.
$$

__Recall__ is the fraction of admits that we say "yes" to:

$$
\mbox{Recall} = \frac{\mbox{True Positives}}{\mbox{True Positives + False Negatives}}.
$$

In [ ]:
#| code-fold: true
def evaluate(y_train, X_train, y_test, X_test, threshold):

    # learn model on training data
    logit = sm.Logit(y_train, X_train)
    result = logit.fit(disp=False)

    # make probability predictions on test data
    y_pred = result.predict(X_test)

    # threshold probabilities to create classifications
    y_pred = y_pred > threshold

    # report metrics
    precision = metrics.precision_score(y_test, y_pred, zero_division=0)
    recall = metrics.recall_score(y_test, y_pred)
    accuracy = metrics.accuracy_score(y_test, y_pred)
    return precision, recall, accuracy

precision, recall, accuracy = evaluate(y_train, X_train, y_test, X_test, 0.5)

print(f'Precision: {precision:0.3f}, Recall: {recall:0.3f}, Accuracy: {accuracy:0.3f}')
print(f'Accuracy of "always reject" on this test set: {1 - y_test.mean():0.3f}')

The accuracy barely beats "always reject", and the recall says why: at a 0.5
threshold the model says *yes* to only a small fraction of the real admits.


A single random split is noisy. Let's average precision and recall over 20
different train/test splits:

In [ ]:
#| code-fold: true
PR = []
for i in range(20):
    X_train, X_test, y_train, y_test = model_selection.train_test_split(
            df[train_cols], df['admit'],
            test_size=0.4, random_state=i)
    PR.append(evaluate(y_train, X_train, y_test, X_test, 0.5))

In [ ]:
#| code-fold: true
avgPrec = np.mean([f[0] for f in PR])
avgRec = np.mean([f[1] for f in PR])
avgAcc = np.mean([f[2] for f in PR])
print(f'Average Precision: {avgPrec:0.3f}, Average Recall: {avgRec:0.3f}, Average Accuracy: {avgAcc:0.3f}')

Sometimes we would like a single value that describes the overall performance of
the classifier.

For this, we take the harmonic mean of precision and recall, called __F1 Score__:

$$
\mbox{F1 Score} = 2 \;\;\frac{\mbox{Precision} \cdot \mbox{Recall}}{\mbox{Precision} + \mbox{Recall}}.
$$


Using this, we can evaluate other settings for the threshold.

In [ ]:
#| code-fold: true
def evalThresh(df, thresh):
    PR = []
    for i in range(20):
        X_train, X_test, y_train, y_test = model_selection.train_test_split(
                df[train_cols], df['admit'],
                test_size=0.4, random_state=i)
        PR.append(evaluate(y_train, X_train, y_test, X_test, thresh))
    avgPrec = np.mean([f[0] for f in PR])
    avgRec = np.mean([f[1] for f in PR])
    # at high thresholds the model predicts no positives at all: F1 is 0, not 0/0
    f1 = 0.0 if avgPrec + avgRec == 0 else 2 * (avgPrec * avgRec) / (avgPrec + avgRec)
    return f1, avgPrec, avgRec

tvals = np.linspace(0.05, 0.8, 50)
f1vals = [evalThresh(df, tval)[0] for tval in tvals]

In [ ]:
#| code-fold: true
#| fig-align: center
plt.figure(figsize=(6, 3))
plt.plot(tvals,f1vals)
plt.ylabel('F1 Score')
plt.xlabel('Threshold for Classification')
plt.title('F1 as a function of Threshold');

Based on this plot, we can say that the best classification threshold appears to
be around 0.35 -- well below 0.5 -- where precision and recall are:

In [ ]:
#| code-fold: true
F1, Prec, Rec = evalThresh(df, 0.35)
print('Precision: {:0.3f}, Recall: {:0.3f}, F1: {:0.3f}'.format(Prec, Rec, F1))

Compared with the 0.5 threshold we traded precision for recall: the model now
says *yes* to most of the real admits, at the cost of more false alarms.


$0.5$ is not a law of nature: it is where the model thinks the two classes are
equally likely, which says nothing about what the two kinds of mistake *cost*.

If a false positive costs $C_{FP}$ and a false negative costs $C_{FN}$, saying
"yes" is worth it exactly when $p\,C_{FN} > (1-p)\,C_{FP}$, i.e. when

$$
p > t^* = \frac{C_{FP}}{C_{FP} + C_{FN}}.
$$

A miss that is four times as costly as a false alarm puts the threshold at
$t^* = 0.2$. Choosing $t$ is a decision about costs, and F1 is just one
cost-free way to summarize the trade-off.


The example here is based on
http://blog.yhathq.com/posts/logistic-regression-and-python.html
where you can find additional details.


## ROC-AUC Curve


Two rates, both out of the *true* classes:

* **True positive rate** (TPR, = recall): $\text{TP}/(\text{TP}+\text{FN})$,
  the fraction of real admits we say yes to.
* **False positive rate** (FPR): $\text{FP}/(\text{FP}+\text{TN})$,
  the fraction of real rejects we wrongly say yes to.

* The **ROC curve** plots TPR against FPR, one point per threshold.
  The diagonal is a random classifier.

* The **AUC** is the area under the ROC curve: a threshold-free summary of how
  well the model *ranks* admits above rejects.


In [ ]:
#| code-fold: true
#| fig-align: center
# Fit on a training split and score the held-out split -- a ROC curve on the
# training data would flatter the model.
X_train, X_test, y_train, y_test = model_selection.train_test_split(
        df[train_cols], df['admit'], test_size=0.4, random_state=1)
roc_fit = sm.Logit(y_train, X_train).fit(disp=False)
y_score = roc_fit.predict(X_test)

fpr, tpr, thresholds = metrics.roc_curve(y_test, y_score)
auc_score = metrics.roc_auc_score(y_test, y_score)

fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(fpr, tpr, linewidth=2, label=f'ROC curve (AUC = {auc_score:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC curve on the held-out split', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_score:.4f}")

## From logistic regression to regularization

* We can now fit **linear** models (last lecture) and **logistic** models (just now), both by choosing coefficients $\boldsymbol{\beta}$ that minimize a loss.
* Both can **overfit**, exactly as [Generalization](07-Generalization.qmd) warned: with many (or correlated) features, the fitted $\boldsymbol{\beta}$ chases noise in the training set and generalizes poorly.
* **Regularization** is the fix -- add a penalty on the size of $\boldsymbol{\beta}$ to the loss -- and it is the *same idea* for linear regression, logistic regression, and later neural networks.
* Part 2 develops it for linear regression, where the geometry is easiest to see, and closes by applying it to the logistic model we just fit.


# Part 2: Regularization

In [ ]:
#| echo: false
import warnings

import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

np.random.seed(9876789)

## Overfitting

We have referenced the concept of overfitting previously in our [generalization](07-Generalization.qmd) and [decision tree](08-Classification-I-Decision-Trees.qmd) lectures.

We know that a model that overfits does not generalize well. In other words, the model does not perform well on data it was not trained on.

We observe overfitting when the training accuracy of the model is high, while the accuracy on the validation set (or held out test set) stagnates.

In this part of the lecture we will discuss **regularization**. A technique to help prevent overfitting. We will see how to apply regularization to regression problems.

However, regularization is a general technique to ensure models learn broad patterns and is applicable in neural networks.


## What is regularization?

We know regularization is a way to prevent overfitting, but how does this actually work?

The idea behind regularization is to penalize a model's loss function during training. The added penalty will discourage the model from becoming too complex (i.e., overfitting).

In regression, our training process was to compute coefficients $\boldsymbol{\beta}$ by minimizing

$$
\min_{\boldsymbol{\beta}} \Vert X \boldsymbol{\beta} - \mathbf{y}\Vert_{2}^{2},
$$

where, as in the [linear regression](10-Regression-I-Linear.qmd) lecture,
$X\in\mathbb{R}^{n\times p}$ is the design matrix with one row per observation
($n$ observations, $p$ features) and $\mathbf{y}\in\mathbb{R}^{n}$ holds the
dependent variable.


Regularization adds a function $R(\boldsymbol{\beta})$ to the minimization problem. A regularized minimization problem is then: compute $\boldsymbol{\beta}$ such that

$$
\min_{\boldsymbol{\beta}} \Vert X \boldsymbol{\beta} - \mathbf{y}\Vert_{2}^{2} + cR(\boldsymbol{\beta}).
$$

The regularization coefficient $c$ is a hyperparameter that controls the importance of the regularization term $R(\boldsymbol{\beta})$.

We will consider two common forms of $R(\boldsymbol{\beta})$:

- $R(\boldsymbol{\beta}) = \Vert \boldsymbol{\beta}\Vert_{2}^{2}$, called **ridge regression**, and
- $R(\boldsymbol{\beta}) = \Vert \boldsymbol{\beta}\Vert_1$, called **LASSO** regression.


In this part of the lecture we will cover:

- the situation where regression needs help -- **multicollinearity** -- and how
  the **condition number** diagnoses it,
- ridge and LASSO: what each penalty does to the coefficients and when to use which,
- the two rules that come with any penalty: **standardize first**, and choose the
  hyperparameter $c$ by **cross-validation**,
- and finally the same penalty applied to the logistic model from Part 1.


## Multicollinearity

>In statistics, multicollinearity (also collinearity) is a phenomenon in which two or more predictor variables in a multiple regression model are highly correlated, meaning that one can be linearly predicted from the others with a substantial degree of accuracy.

We will see that multicollinearity can be problematic in our regression models.

To address these issues, we will consider

- What are the potential sources for multicollinearity?
- What does multicollinearity tell us about our data?

Understanding these questions will inform us how regularization can be used to mitigate the issue of multicollinearity.


## Sources of Multicollinearity

We will be working with our design matrix $X\in\mathbb{R}^{n\times p}$, where each row corresponds to an observation and the $p$ columns are the $p$ features.

We will see that multicollinearity arises when the columns of $X$ are linearly dependent (or nearly linearly dependent).

As a consequence, the matrix $X^{T}X$ is no longer invertible. However, as we saw in our [Linear Regression](10-Regression-I-Linear.qmd) lecture, the least squares solution still exists. It is just not unique.


What are the situations where this can happen?

One clear case is when $n < p$, which means $X$ has more columns than rows. That is, there are  __more features than there are observations__ in the data.

However, we can still observe multicollinearity when $n > p$.

This can happen when the columns of $X$ happen to be linearly dependent because of the nature of the data itself. In particular, this happens when one column is a linear function of the other columns. This means that one independent variable is a linear function of one or more of the others.

Unfortunately, in practice we will run into trouble even if variables are *almost* linearly dependent.

## The Longley Dataset

To illustrate the multicollinearity problem, we'll load a standard dataset.

The [Longley](https://www.statsmodels.org/stable/datasets/generated/longley.html) dataset contains various US macroeconomic variables from 1947–1962.

We have the following features:

- GNP - GNP (Gross National Product)
- GNPDEFL - GNP deflator
- UNEMP - Number of unemployed
- ARMED - Size of armed forces
- POP - Population
- YEAR - Year (1947 - 1962)

We want to predict:

- TOTEMP - Total Employment


In [ ]:
from statsmodels.datasets.longley import load_pandas
y = load_pandas().endog
X = load_pandas().exog
X['const'] = 1.0
X.index = X['YEAR']
y.index = X['YEAR']
X.drop('YEAR', axis = 1, inplace = True)
print("X.head()")
print(X.head())
print("\n\ny.head()")
print(y.head())

Read the notes under the table: statsmodels warns that the condition number is
large. What does this mean?

In [ ]:
ols_model = sm.OLS(y, X)
ols_results = ols_model.fit()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(ols_results.summary())

## Condition Number

The notion of conditioning pertains to the perturbation behavior of a mathematical problem. A well-conditioned problem is one where small changes to the inputs produce small changes to the output. An ill-conditioned problem is one where small changes in the input can produce very large changes in the output.

The condition number of a matrix provides an indication of how accurately you can compute with it.

A large condition number tells us that our problem is ill conditioned, i.e., small changes to the input can produce very large changes in the output.

A small condition number tells us that our problem is well-conditioned.


The condition number is derived using matrix norms, which is beyond the scope of this course. However, we will use the following definition for the condition number of our matrix

$$
\kappa(X) = \frac{\sigma_{\text{max}}}{\sigma_{\text{min}}},
$$

where $\sigma_{\text{max}}$, $\sigma_{\text{min}}$ are the maximum and minimum singular values, respectively, of the design matrix $X$.

The SVD -- next lecture -- again provides us with important properties of a matrix.


Another important fact is that

$$
\kappa(X^TX) = \frac{\sigma_{\text{max}}^2}{\sigma_{\text{min}}^2} = \kappa(X)^2.
$$

This means that if we have a poorly conditioned data matrix $X$, then $X^TX$ is even more poorly conditioned.

This is why, in practice, you should not *form* $X^TX$ and invert it. The
normal equations describe the solution; good solvers (`np.linalg.lstsq`,
`sm.OLS`) reach the same $\hat{\boldsymbol{\beta}}$ by working with $X$
directly, at condition number $\kappa(X)$ rather than $\kappa(X)^2$.

## Normal Equations: a callback

From the [linear regression](10-Regression-I-Linear.qmd) lecture: the least-squares
solution satisfies the normal equations $X^TX\boldsymbol{\beta} = X^T\mathbf{y}$,
which always have at least one solution.

* The solution is **unique** exactly when $X^TX$ is invertible, i.e. when the
  columns of $X$ are linearly independent.
* If the columns are **linearly dependent**, $X^TX$ has a zero singular value,
  $\kappa(X^TX) = \infty$, and $(X^TX)^{-1}$ does not exist. There are then
  infinitely many $\boldsymbol{\beta}$ with the *same* fitted values
  $X\boldsymbol{\beta}$ and the same residual norm: the **fit** is determined,
  the **coefficients** are not.
* This is the simplest kind of multicollinearity. It is a problem of
  interpretation, not of prediction.


## Near Linear Dependence

Near linear dependence causes problems as well. This can happen, for example, due to  measurement errors. Or when two or more columns are __strongly correlated__.

In such a situation, we have some column of our design matrix that is __close to__ being a linear combination of the other columns.

When these situations occur we will have problems with linear regression.


As a result of near linear dependence, the smallest singular value of the design matrix $X$ will be close to zero. This means that $\kappa(X)$ will be very large.

The condition number tells us that a small change in the input to our problem can result in large changes to the output.

This means that for a design matrix $X$ with near linearly dependent columns, the values we compute for $\boldsymbol{\beta}$ in our linear regression can vary significantly.

Put in the language of [Generalization](07-Generalization.qmd): an unstable
$\hat{\boldsymbol{\beta}}$ *is* **variance**. Refit on a slightly different
sample and the coefficients swing, so each individual fit has chased noise. The
cure for variance there was to constrain the model; here the constraint is a
penalty on $\boldsymbol{\beta}$, which *regularizes* the solution.


## Longley: reading the condition number

Recall that the condition number of our data is around $10^8$.

A large condition number is evidence of a problem.

As a general rule of thumb:

* If the condition number is less than 100, there is no serious problem
with multicollinearity.
* Condition numbers between 100 and 1000 imply moderate to strong multicollinearity.
* Condition numbers bigger than 1000 indicate severe multicollinearity.


The correlation matrix of the features shows where the near-dependence comes from.

In [ ]:
#| fig-align: center
#| code-fold: true
longley_features = ['GNPDEFL', 'GNP', 'UNEMP', 'ARMED', 'POP']
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(X[longley_features].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Longley features: pairwise correlations', fontsize=13)
plt.tight_layout()

We can see __very__ strong linear relationships between __GNP Deflator__, __GNP__, and __Population__ (correlations above 0.97): three columns carrying essentially one piece of information.

## Addressing Multicollinearity

Here are two strategies we can employ to address multicollinearity:

1. Ridge Regression
2. Model Selection via LASSO

PCA also addresses multicollinearity by transforming the correlated features into uncorrelated features. However in this approach you lose the original features, which is less explainable.


## Why the coefficients blow up

The first thing to note is that when columns of $X$ are nearly dependent, the components of $\hat{\boldsymbol{\beta}}$ tend to be __large in magnitude__.

Consider a regression in which we are predicting the point $\mathbf{y}$ as a
linear combination $\beta_1\mathbf{u} + \beta_2\mathbf{v}$ of two $X$ columns
$\mathbf{u}$ and $\mathbf{v}$. In two dimensions this is exact, and the
coefficients are the solution of a $2\times 2$ system, computed below with
`np.linalg.solve`.

In [ ]:
#| code-fold: true
#| fig-align: center
def arrow(ax, start, end, color='k', lw=1.8):
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                                shrinkA=0, shrinkB=0))

def decompose(ax, u, v, y, show_components, title, u_label_offset):
    beta = np.linalg.solve(np.column_stack([u, v]), y)   # y = beta1 u + beta2 v
    b1u = beta[0] * u
    arrow(ax, (0, 0), u); ax.text(*(u + u_label_offset), r'$\mathbf{u}$', fontsize=13)
    arrow(ax, (0, 0), v); ax.text(*(v + [0.2, 0.2]), r'$\mathbf{v}$', fontsize=13)
    ax.plot(*y, 'ro'); ax.text(*(y + [-0.9, 0.3]), r'$\mathbf{y}$', fontsize=13)
    if show_components:
        arrow(ax, (0, 0), b1u, color='tab:blue')
        ax.text(*(b1u * 0.75 + [0.4, -0.2]), r'$\beta_1\mathbf{u}$', color='tab:blue', fontsize=13)
        arrow(ax, b1u, y, color='tab:red')
        ax.text(*((b1u + y) / 2 + [-0.3, 0.4]), r'$\beta_2\mathbf{v}$', color='tab:red', fontsize=13)
    ax.set_title(title + rf'   $\beta = ({beta[0]:.2f},\ {beta[1]:.2f})$', fontsize=12)
    return beta

y_target = np.array([-3.4, 1.95])
u_apart, v_apart = np.array([1.0, 2.0]), np.array([4.0, 1.0])   # well separated
u_close, v_close = np.array([2.0, 1.0]), np.array([4.0, 1.0])   # nearly dependent

fig, axs = plt.subplots(2, 2, figsize=(11, 6.5))
for row, (u, v, name, off) in enumerate([(u_apart, v_apart, 'well separated:', [0.3, -0.3]),
                                         (u_close, v_close, 'nearly dependent:', [-0.4, -0.9])]):
    for col in range(2):
        ax = axs[row, col]
        ax.set_xlim(-6, 13); ax.set_ylim(-1.5, 7)
        ax.set_aspect('equal')
        ut.centerAxes(ax)
        ax.set_xticks([]); ax.set_yticks([])
        decompose(ax, u, v, y_target, show_components=(col == 1), title=name, u_label_offset=off)
plt.tight_layout()

* **Top row:** $\mathbf{u}$ and $\mathbf{v}$ point in clearly different
  directions, and the same $\mathbf{y}$ is reached with coefficients of modest
  size, $\boldsymbol{\beta} = (1.60, -1.25)$.
* **Bottom row:** $\mathbf{u}$ and $\mathbf{v}$ are nearly parallel. The part
  of $\mathbf{y}$ that points *away* from their shared direction can only be
  built as the difference of two long, nearly cancelling vectors:
  $\boldsymbol{\beta} = (5.60, -3.65)$, more than three times as large for the
  same target.
* Push $\mathbf{u}$ and $\mathbf{v}$ closer still and the coefficients grow
  without bound -- and flip wildly if $\mathbf{y}$ moves a little.


This geometric argument illustrates why the regression coefficients will be very
large under multicollinearity. As a result, $\Vert\boldsymbol{\beta}\Vert_2$
will be very large -- which is exactly what ridge regression penalizes.


## Ridge Regression

Ridge regression adjusts the least squares regression by shrinking the estimated coefficients towards zero.

The purpose is to fix the magnitude inflation of $\Vert\boldsymbol{\beta}\Vert_2$.

To do this, Ridge regression assumes that the model has no intercept term -- both the response and the predictors have been centered so that $\beta_0 = 0$.

Ridge regression then consists of adding a penalty term to the regression:

$$
\hat{\boldsymbol{\beta}} = \arg \min_\boldsymbol{\beta} \Vert X\boldsymbol{\beta} - \mathbf{y} \Vert_2^2 + c\Vert\boldsymbol{\beta}\Vert_2^2.
$$


For any given $c$ this has a closed-form solution in which $\hat{\boldsymbol{\beta}} = (X^TX +cI)^{−1}X^T\mathbf{y}.$

The solution to the Ridge regression problem always exists and is unique, even when the data contains multicollinearity: adding $c$ to every eigenvalue of $X^TX$ lifts the smallest one off zero.

Here, $c \geq 0$ is a tradeoff parameter and controls the strength of the penalty term:

* When $c = 0$, we get the least squares estimator: $\hat{\boldsymbol{\beta}} = (X^TX)^{−1}X^T\mathbf{y}$
* When $c \rightarrow \infty$, we get $\hat{\boldsymbol{\beta}} \rightarrow 0.$
* Increasing the value of $c$ forces the norm of $\hat{\boldsymbol{\beta}}$ to decrease, yielding smaller coefficient estimates in magnitude.

For a finite, positive value of $c$, we are balancing two tasks: fitting
a linear model and shrinking the coefficients.

The coefficient $c$ is a __hyperparameter__ that controls the model complexity. We typically set $c$ by holding out data, i.e., __cross-validation.__

## Ridge in one dimension

With a single (centered) column $\mathbf{x}$, write $s = \mathbf{x}^T\mathbf{x}$
and let $\hat\beta_{LS} = \mathbf{x}^T\mathbf{y}/s$ be the least-squares slope.
Expanding the squared error around it,
$\Vert\mathbf{y} - \mathbf{x}\beta\Vert_2^2 = s\,(\beta - \hat\beta_{LS})^2 + \text{const}$,
so ridge minimizes

$$
s\,(\beta - \hat\beta_{LS})^2 + c\,\beta^2
\quad\Longrightarrow\quad
2s(\beta - \hat\beta_{LS}) + 2c\beta = 0
\quad\Longrightarrow\quad
\hat\beta_{\text{ridge}} = \frac{s}{s + c}\,\hat\beta_{LS} = \frac{\hat\beta_{LS}}{1 + c/s}.
$$

* Ridge **multiplies** the least-squares estimate by a factor in $(0, 1)$: it shrinks, but never reaches zero for finite $c$.
* For a standardized column $s = \mathbf{x}^T\mathbf{x} = n$, so the shrinkage factor is $1/(1 + c/n)$: a penalty of $c = n$ halves every coefficient. (Keep this in mind when reading the $c$ axis of the plots below -- $c$ is on the scale of $n$.)
* In many dimensions the same thing happens along each singular direction of $X$, with factor $\sigma_j^2/(\sigma_j^2 + c)$: the directions with tiny $\sigma_j$ -- the nearly dependent ones -- are shrunk the most. That is the next lecture's story.


## Scaling

Note that the penalty term $\Vert\boldsymbol{\beta}\Vert_2^2$ would be unfair to the different predictors if they are not on the same scale: a useful feature measured in tiny units needs a huge coefficient and is punished for it, while a feature in huge units is nearly free.

Therefore, if we know that the variables are not measured in the same units, we first standardize the columns of $X$ (zero mean, unit variance) and center $\mathbf{y}$, and then perform ridge regression.

By centering $\mathbf{y}$ and $X$ we do not need (or include) an intercept in the model -- and the intercept should never be penalized.

**Fit the scaler on the training split only.** The means and standard deviations are
part of the model; computing them on data you will later test on leaks test
information into training.

Another name for ridge regression is __Tikhonov regularization__. You may see this terminology used in textbooks on optimization.

Standardizing is needed for this specific method. This is in contrast to the [linear regression](10-Regression-I-Linear.qmd) lecture where we allowed the coefficients to correct for the scaling differences between different units of measure.


## Ames housing: a data set for the sweeps

Longley has only 16 rows -- fine for showing a condition number, hopeless for
cross-validation (8 training points for 5 features). For the penalty sweeps we
switch to the Ames housing data from the [linear regression](10-Regression-I-Linear.qmd)
lecture: 1460 houses, target `SalePrice`, and eight numeric features chosen to
include three **genuinely collinear pairs**.

In [ ]:
#| code-fold: true
ames = pd.read_csv('data/ames-housing-data/train.csv')
ames_features = ['OverallQual', 'YearBuilt',
                 'GrLivArea', 'TotRmsAbvGrd',      # living area  vs  number of rooms
                 'GarageCars', 'GarageArea',       # garage capacity two ways
                 '1stFlrSF', 'TotalBsmtSF']        # first floor sits on the basement
X_ames = ames[ames_features]
y_ames = ames['SalePrice']
print(X_ames.shape)
X_ames.describe().loc[['mean', 'std', 'min', 'max']].round(0)

In [ ]:
#| code-fold: true
#| fig-align: center
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(X_ames.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Ames features: pairwise correlations', fontsize=13)
plt.tight_layout()

The three pairs correlate at 0.83, 0.88 and 0.82. Nothing here is *exactly*
dependent, so least squares runs -- but, as the geometry predicted, it has to
decide how to split one effect between two near-duplicate columns, and that
decision is unstable.


For each $c$ we repeat 100 random train/test splits. To make the variance of
$\hat{\boldsymbol{\beta}}$ -- the thing the penalty controls -- visible, we
train on a deliberately small 20% split (292 houses) and test on the other 80%.
The scaler is fit on the training split each time. We use `sklearn`'s `Ridge`,
whose `alpha` is exactly our $c$.

In [ ]:
#| code-fold: true
def heldout_sweep(make_model, c_values, X, y, nreps=100, test_size=0.8):
    """Mean held-out R^2 (with standard error) and mean coefficients for each c."""
    scores, coefs = [], []
    for c in c_values:
        r2s, betas = [], []
        for rep in range(nreps):
            X_tr, X_te, y_tr, y_te = model_selection.train_test_split(
                X, y, test_size=test_size, random_state=rep)
            x_scaler = StandardScaler().fit(X_tr)                      # training split only
            y_scaler = StandardScaler().fit(y_tr.to_frame())
            X_tr_s, X_te_s = x_scaler.transform(X_tr), x_scaler.transform(X_te)
            y_tr_s = y_scaler.transform(y_tr.to_frame()).ravel()
            y_te_s = y_scaler.transform(y_te.to_frame()).ravel()
            model = make_model(c, len(y_tr_s)).fit(X_tr_s, y_tr_s)
            r2s.append(r2_score(y_te_s, model.predict(X_te_s)))
            betas.append(model.coef_)
        scores.append([c, np.mean(r2s), np.std(r2s) / np.sqrt(nreps)])
        coefs.append(np.mean(betas, axis=0))
    return np.array(scores), pd.DataFrame(coefs, index=c_values, columns=X.columns)

def plot_sweep(scores, label, title):
    """scores[0] is the unpenalized (c = 0) baseline; the rest are plotted vs log10(c).
    Left: the whole sweep. Right: zoomed in on the top, where the gain over c = 0 lives."""
    base, sweep = scores[0], scores[1:]
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    span = [np.log10(sweep[0, 0]), np.log10(sweep[-1, 0])]
    for ax in axs:
        ax.errorbar(np.log10(sweep[:, 0]), sweep[:, 1], sweep[:, 2], marker='o', ms=3,
                    capsize=2, label=label)
        ax.hlines(base[1], *span, linestyles='dashed', label='Without regularization (c = 0)')
        ax.hlines([base[1] - base[2], base[1] + base[2]], *span, linestyles='dotted',
                  label=r'$\pm 1$ standard error')
        ax.set_xlabel(r'$\log_{10}(c)$', fontsize=13)
        ax.tick_params(labelsize=11)
    axs[0].set_ylabel('held-out $R^2$', fontsize=13)
    axs[0].set_title('full sweep', fontsize=12)
    axs[0].legend(loc='lower left', fontsize=9)
    axs[1].set_ylim(base[1] - 0.02, sweep[:, 1].max() + 0.006)
    axs[1].set_title('zoom on the top', fontsize=12)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    return axs

def ridge_model(c, n_train):
    return Ridge(alpha=c, fit_intercept=False) if c > 0 else LinearRegression(fit_intercept=False)

ridge_cs = np.r_[0, 10 ** np.linspace(-1, 4, 21)]
ridge_scores, ridge_coefs = heldout_sweep(ridge_model, ridge_cs, X_ames, y_ames)

In [ ]:
#| code-fold: true
#| fig-align: center
plot_sweep(ridge_scores, 'Ridge regression', 'Ridge on Ames housing: held-out $R^2$ vs. $c$')
plt.show()

* For small $c$ nothing changes: the penalty is negligible next to the squared error.
* Around $c \approx 10$–$100$ (recall $n = 292$, so this is shrinkage of roughly
  5–25%) the held-out $R^2$ rises **above** the unpenalized fit, by several
  standard errors. Shrinking the coefficients made the predictions *better*: bias
  bought, variance sold.
* Past $c \approx 300$ the shrinkage is too strong, the model underfits and $R^2$
  collapses toward zero.
* The sweet spot is found exactly this way -- on **held-out** data. Training
  error is always lowest at $c = 0$; the penalty can only make the training fit worse.



To sum up the idea behind Ridge regression:

1. There may be many $\boldsymbol{\beta}$ values that are consistent with the equations.
1. Over-fit $\boldsymbol{\beta}$ values tend to have large magnitudes.
1. We add the regularization term $c \Vert \boldsymbol{\beta}\Vert_2^2$ to the least squares to avoid those solutions.
1. We tune $c$ to an appropriate value via cross-validation.

## Model Selection

Of course, one might attack the problem of multicollinearity as follows:

- Multicollinearity occurs when variables (features) are close to linearly dependent.
- These variables do not contribute anything *meaningful* to the quality of the model
- As a result why not simply remove variables from the model that are nearly linearly dependent?

We create a new model when we remove these variables from our regression.

This strategy is called **model selection**.


Removing columns is easy to *do*. With statsmodels' formula interface a model is
a one-line, R-style formula -- `response ~ feature + feature` -- and the
intercept is included for you:

In [ ]:
X['TOTEMP'] = y
reduced = smf.ols('TOTEMP ~ GNPDEFL + GNP', data=X).fit()
reduced.summary().tables[1]

Keeping only GNP and the GNP deflator does not help: the two are correlated at
0.99, and the data cannot tell their effects apart -- the deflator's interval
runs from $-352$ to $+182$. We would have to try another subset, and another.
Formula syntax (categorical variables, transformations, dropping the intercept)
is covered in [Appendix: Regression Formulas with statsmodels](A8-Statsmodels-Formulas.qmd).


One of the advantages of model selection is __interpretability__: by eliminating variables, we get a clearer picture of the relationship between truly useful features and dependent variables.

However, there is a big challenge inherent in model selection. In general, the possibilities to consider are exponential in the number of features.

That is, if we have $p$ features to consider, then there are $2^p-1$ possible models that incorporate one or more of those features. This space is usually too big to search directly -- and we just saw that picking a subset by eye is not reliable either.

Can we use Ridge regression for this problem?

Ridge regression does not set any coefficients exactly to zero unless $c\rightarrow \infty$, in which case they're all zero.

This means Ridge regression cannot perform variable selection. Even though it performs well in terms of prediction accuracy, it does not offer a clear interpretation.



## The LASSO

LASSO differs from Ridge regression __only in terms of the norm__ used by the penalty term.

$$
\hat{\boldsymbol{\beta}} = \arg \min_{\boldsymbol{\beta}} \Vert X\boldsymbol{\beta} - \mathbf{y} \Vert_2^2 + c \Vert\boldsymbol{\beta}\Vert_1.
$$

However, this small change in the norm makes a __big difference__ in practice.

The nature of the $\ell_1$ penalty will cause some coefficients to be shrunken to zero exactly.

This means that LASSO can perform model selection by telling us which variables to keep and which to set aside.

As $c$ increases, more coefficients are set to zero, i.e., fewer variables are selected.

In terms of prediction error, LASSO performs comparably to Ridge regression but it has a __big advantage with respect to interpretation.__

## Why $\ell_1$ gives exact zeros: the soft threshold

Repeat the one-dimensional calculation with the $\ell_1$ penalty: minimize
$s\,(\beta - \hat\beta_{LS})^2 + c\,|\beta|$. The derivative of $|\beta|$ is
$\pm 1$ away from zero, so for $\beta > 0$ we need
$2s(\beta - \hat\beta_{LS}) + c = 0$, i.e. $\beta = \hat\beta_{LS} - c/(2s)$ --
which is only valid if the result is still positive. Symmetrically for $\beta < 0$. Otherwise the minimum is at the kink, $\beta = 0$:

$$
\hat\beta_{\text{lasso}} = \operatorname{sign}(\hat\beta_{LS})\,\max\!\big(|\hat\beta_{LS}| - \tfrac{c}{2s},\ 0\big).
$$

* Ridge **multiplied** by a factor: a small coefficient stays small but nonzero.
* LASSO **subtracts** a fixed amount $c/(2s)$ and stops at zero: every coefficient
  whose least-squares value is smaller than $c/(2s)$ is set *exactly* to zero.
  That is why the $\ell_1$ penalty selects variables.



![](figs/L11-recap-l1-l2-geometry.png)

Minimizing the penalized loss is equivalent to minimizing the squared error
subject to $R(\boldsymbol{\beta}) \le$ some budget: the ellipses are contours of
the squared error, the shaded region is the penalty ball. The $\ell_2$ ball is
round, so the contours touch it at a generic point with both coefficients
nonzero. The $\ell_1$ ball is a diamond with **corners on the axes**, and the
contours usually hit a corner first -- a corner means a coefficient is exactly zero.


Same protocol as for ridge. `sklearn`'s `Lasso` minimizes
$\frac{1}{2n}\Vert X\boldsymbol{\beta}-\mathbf{y}\Vert_2^2 + \alpha\Vert\boldsymbol{\beta}\Vert_1$,
so its `alpha` is $c/(2n)$; we convert so that the $c$ axis means the same thing in
both plots.

In [ ]:
#| code-fold: true
def lasso_model(c, n_train):
    if c == 0:
        return LinearRegression(fit_intercept=False)
    return Lasso(alpha=c / (2 * n_train), fit_intercept=False, max_iter=20000)

lasso_cs = np.r_[0, 10 ** np.linspace(-1, 3, 25)]
lasso_scores, lasso_coefs = heldout_sweep(lasso_model, lasso_cs, X_ames, y_ames)

In [ ]:
#| code-fold: true
#| fig-align: center
plot_sweep(lasso_scores, 'LASSO regression', 'LASSO on Ames housing: held-out $R^2$ vs. $c$')
plt.show()

The shape is the same as for ridge -- flat, a modest gain, then collapse -- but
the collapse comes earlier and is sharper: once $c$ is large enough to zero the
important features, there is nothing left to predict with.


The interesting output is not the $R^2$ curve but **which coefficients survive**.
For the path we fit the LASSO once on all 1460 (standardized) houses for each $c$,
so the zeros are exact rather than averaged away. Each line is one feature's
coefficient as $c$ grows.

In [ ]:
#| code-fold: true
#| fig-align: center
X_all = StandardScaler().fit_transform(X_ames)
y_all = StandardScaler().fit_transform(y_ames.to_frame()).ravel()
n_all = len(y_all)

path_cs = 10 ** np.linspace(-1, 3.5, 28)
lasso_path = pd.DataFrame(
    [Lasso(alpha=c / (2 * n_all), fit_intercept=False, max_iter=20000).fit(X_all, y_all).coef_
     for c in path_cs],
    index=path_cs, columns=ames_features)

ax = lasso_path.set_index(np.log10(path_cs)).plot(figsize=(8, 4.5), linewidth=2)
ax.axhline(0, color='k', linewidth=0.8)
ax.set_xlabel(r'$\log_{10}(c)$', fontsize=13)
ax.set_ylabel('coefficient (standardized)', fontsize=13)
ax.set_title('LASSO coefficient path on Ames housing (all 1460 houses)', fontsize=14)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=10)
ax.tick_params(labelsize=11)
plt.tight_layout()

**Why the $c$ axis differs from the sweep**

The one-dimensional formulas say the penalty acts through $c/n$. The sweep
trained on $n = 292$ houses; the path uses all $1460$, so the same $c$ is five
times gentler here. Read the path for the *order* in which coefficients die, not
for where the $c$ values line up with the $R^2$ plot.


In [ ]:
#| code-fold: true
# the coefficients at a few values of c, for reading the path
show_cs = [0.1, 10, 100, 1000]
lasso_path.loc[[c for c in path_cs if any(np.isclose(c, s, rtol=1e-2) for s in show_cs)]].round(3).T

* With almost no penalty `TotRmsAbvGrd` has a **negative** coefficient: "more
  rooms, lower price, holding living area fixed". That sign is an artefact of
  splitting one effect between two near-duplicate columns -- the geometry from
  earlier, in data.
* It is the **first coefficient LASSO zeroes** (by $c \approx 5$). Of the pair
  `GrLivArea` / `TotRmsAbvGrd`, LASSO keeps the square footage.
* The remaining coefficients shrink slowly until $c$ is in the hundreds, then
  `YearBuilt` goes, followed by `GarageArea` and `1stFlrSF` -- the weaker member
  of each remaining pair. `GarageCars` and `TotalBsmtSF` survive them.
* By $c \approx 1000$ two features carry the model, `OverallQual` and `GrLivArea`,
  with a little `GarageCars` and `TotalBsmtSF` left.


LASSO did model selection for us: from each collinear pair it kept one column,
and it dropped them in the order of how little they added. Ridge would have
shrunk every one of these coefficients and set none to zero.


## Regularizing the logistic model

The penalty is not special to least squares. Add it to the negative
log-likelihood of Part 1 and you have **regularized logistic regression**:

$$
\min_{\boldsymbol{\beta}}\; -\ell(\boldsymbol{\beta}) + c\Vert\boldsymbol{\beta}\Vert_2^2
\qquad\text{or}\qquad
-\ell(\boldsymbol{\beta}) + c\Vert\boldsymbol{\beta}\Vert_1 .
$$

`sklearn`'s `LogisticRegression` does this **by default**: its parameter `C` is the
*inverse* penalty, $C = 1/c$, and the default is `C=1.0`, i.e. an $\ell_2$ penalty
with $c = 1$. Setting `C=np.inf` (older versions: `penalty=None`) gives the plain
maximum-likelihood fit of Part 1.


Standardize the three admissions features (the penalty needs it), then fit with no
penalty, with the default, and with a strong penalty:

In [ ]:
#| code-fold: true
X_adm = df[['gre', 'gpa', 'rank']]
X_adm_s = StandardScaler().fit_transform(X_adm)

fits = {
    'no penalty: C=inf': LogisticRegression(C=np.inf),   # older sklearn: penalty=None
    'default: C=1  (c=1)': LogisticRegression(),
    'C=0.01  (c=100)': LogisticRegression(C=0.01),
}
logit_coefs = pd.DataFrame(
    {name: model.fit(X_adm_s, df['admit']).coef_[0] for name, model in fits.items()},
    index=X_adm.columns)
logit_coefs.round(3)

* With $n = 400$ and three well-conditioned, standardized features, the default
  penalty barely moves the coefficients -- the data dominate $c = 1$.
* Turn the penalty up ($c = 100$) and every coefficient is shrunk toward zero,
  roughly halved; the model becomes less confident everywhere.
* The practical point: **unless you ask for `penalty=None`, sklearn is
  regularizing**, and `C` is $1/c$ -- *smaller* `C` means *more* penalty. In today's
  activity you will meet this in the first `LogisticRegression()` you fit.


## Recap: what you can now do

1. **Write the logistic model** -- a linear function $\boldsymbol{\beta}^T\mathbf{x}$ equal to the
   log-odds, inverted through the sigmoid $\sigma$ -- read $\beta_j$ as a log-odds slope and
   $e^{\beta_j}$ as an odds ratio, and explain why the decision boundary is still a
   hyperplane (the nonlinearity lives in the confidence).
2. **Fit it by maximum likelihood:** write the log-likelihood (minus the log-loss),
   use $\sigma' = \sigma(1-\sigma)$ to get the gradient $X^T(\mathbf{y}-\mathbf{p})$, take
   one gradient step by hand, and say why there is no closed form.
3. **Explain why near-dependent columns make coefficients large and unstable** (huge
   $\kappa(X)$) and how penalties fix it: ridge shrinks by $1/(1 + c/s)$, LASSO
   soft-thresholds to exact zeros, standardize before penalizing, choose $c$ on
   held-out data.
4. **Evaluate a classifier beyond accuracy:** accuracy has a floor set by prevalence;
   sweep the threshold, trade precision against recall and F1, and read ROC/AUC
   for what it does (ranking) and does not (choose a threshold) tell you.
5. **Choose a threshold to ship:** the cost-optimal threshold is
   $C_{FP}/(C_{FP}+C_{FN})$ -- in today's activity you will apply it to an
   imbalanced problem where accuracy lies, and compare it with `class_weight="balanced"`.
